# V67 - User-kNN collaborative smoothing

Pre-registered plan: every tested model is parametric, but the data supports a memory-based route that no version tried: predict a user by similarity-weighted train labels. Item-kNN is impossible (test users have no history), user-kNN is not. Two similarity spaces (sparse word plus assessment, and E5 dense), k fixed at 50 up front, neighbors restricted to training folds under the standard grouped protocol. Gate against word-only Ridge with the usual locks.

In [ ]:
import re
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import Ridge
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics.pairwise import cosine_similarity
import json

SEED = 717
N_SPLITS = 5
GATE_FLOOR = 0.002
K = 50
rng = np.random.RandomState(SEED)
WORDS = {"nol": 0, "satu": 1, "dua": 2, "tiga": 3, "empat": 4, "lima": 5, "tidak": 0, "ya": 5}

CANDIDATES = [Path("/kaggle/input/intelligo-data/data"), Path("data")]
DATA = next((p for p in CANDIDATES if (p / "train_relevance.csv").exists()), None)
assert DATA is not None, "train_relevance.csv not found"
OUT = Path("/kaggle/working" if Path("/kaggle/working").exists() else "results/v67")
OUT.mkdir(parents=True, exist_ok=True)
print("DATA:", DATA.resolve())

## Similarities and fold-restricted neighbor predictions

Cosine similarities are precomputed once. Each validation user sees only training-fold neighbors, so no label reuse. Self-similarity is excluded.

In [ ]:
MODS = [f"M_{i:03d}" for i in range(1, 18)]
train = pd.read_csv(DATA / "train_relevance.csv")
test_ids = pd.read_csv(DATA / "test.csv")["user_id"].tolist()
chats = pd.read_csv(DATA / "chat_history.csv")
assess = pd.read_csv(DATA / "user_assessments.csv")
assert not (set(train["user_id"]) & set(test_ids))
Y = train[MODS].to_numpy(float)
n_train = len(train)
all_users = list(train["user_id"]) + test_ids

def num(x):
    try:
        return float(x)
    except (TypeError, ValueError):
        return float(WORDS.get(str(x).strip().lower(), np.nan))

blobs = {r["user_id"]: json.loads(r["assessment_result"]) for _, r in assess.iterrows()}
questions = sorted({q for b in blobs.values() for q in b})
A = np.array([[num(blobs[u].get(q)) for q in questions] for u in all_users])
A = np.where(np.isnan(A), np.nanmean(A, axis=0), A)
A = (A - A.mean(0)) / (A.std(0) + 1e-9)
by_user = chats.groupby("user_id")["user_chat_text"].apply(list)
docs = [" ".join(str(t) for t in by_user.get(u, [])) for u in all_users]
W = TfidfVectorizer(max_features=5000, ngram_range=(1, 1), min_df=3).fit_transform(docs).toarray()
F = np.hstack([W / (np.linalg.norm(W, axis=1, keepdims=True) + 1e-9), A])
E = None
for p in [Path("results/v58/e5_users.npy"), Path("/kaggle/input/intelligo-e5/e5_users.npy")]:
    if p.exists():
        E = np.load(p)
        print("E5 cache:", p)
        break
assert E is not None, "E5 cache required for the dense variant"

def template_group(texts):
    if not texts:
        return "NO_CHAT"
    return re.sub(r"\d+", "#", " ".join(str(texts[0]).lower().split()[:6]))
fold_groups = np.array([template_group(by_user.get(u, [])) for u in train["user_id"]])
strata = Y.argmax(1)

def ndcg5(yt, ys):
    order = np.argsort(-ys, axis=1)
    rel = np.take_along_axis(yt, order, 1)[:, :5]
    disc = 1.0 / np.log2(np.arange(2, 7))
    dcg = (rel * disc).sum(1)
    ideal = np.sort(yt, axis=1)[:, ::-1][:, :5]
    idcg = (ideal * disc).sum(1)
    return np.where(idcg > 0, dcg / np.maximum(idcg, 1e-12), 0.0)

def knn_predict(S, idx_q, idx_pool, k=K):
    P = np.zeros((len(idx_q), len(MODS)))
    for qi, u in enumerate(idx_q):
        sims = S[u, idx_pool].copy()
        sims[idx_pool == u] = -np.inf
        top = idx_pool[np.argsort(-sims)[:k]]
        w = np.maximum(S[u, top], 0)
        w = w / (w.sum() + 1e-9)
        P[qi] = w @ Y[top]
    return P

S_sparse = cosine_similarity(F[:n_train])
S_dense = cosine_similarity(E[:n_train])
oof_s, oof_d = np.zeros_like(Y), np.zeros_like(Y)
fids = np.zeros(n_train, int)
skf = StratifiedGroupKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
for fi, (tr, va) in enumerate(skf.split(F[:n_train], strata, fold_groups)):
    oof_s[va] = knn_predict(S_sparse, va, tr)
    oof_d[va] = knn_predict(S_dense, va, tr)
    fids[va] = fi + 1
    print(f"fold {fi + 1} done", flush=True)

oof_w = np.zeros_like(Y)
for fi in range(1, N_SPLITS + 1):
    tr, va = np.where(fids != fi)[0], np.where(fids == fi)[0]
    for j in range(len(MODS)):
        m = Ridge(alpha=1.0)
        m.fit(W[:n_train][tr], Y[tr, j])
        oof_w[va, j] = m.predict(W[:n_train][va])
base = ndcg5(Y, oof_w)
print(f"baseline word-only: {base.mean():.5f}")
for name, oo in [("knn sparse", oof_s), ("knn E5", oof_d)]:
    s = ndcg5(Y, oo)
    d = s - base
    lo, hi = np.quantile([rng.choice(d, len(d), replace=True).mean() for _ in range(2000)], [0.025, 0.975])
    wf = min(ndcg5(Y[fids == k], oo[fids == k]).mean() - base[fids == k].mean() for k in range(1, N_SPLITS + 1))
    print(f"{name}: OOF={s.mean():.5f} delta={d.mean():+.5f} CI95=[{lo:+.5f},{hi:+.5f}] worst-fold={wf:+.5f} {'ADOPT' if (lo > GATE_FLOOR and wf >= 0) else 'reject'}")